# NABOPB: a first look

**NABOPB** approximates a high-dimensional function by finding the few basis coefficients
that actually matter, without ever building the full coefficient grid.
Kämmerer, Potts & Taubert, 2023
([doi:10.1007/s43670-023-00057-7](https://doi.org/10.1007/s43670-023-00057-7)).

This notebook covers the core loop only. We take a function whose coefficients we already
know, and watch the algorithm rediscover them one dimension at a time.

1. Why a `d`-dimensional FFT is not an option
2. A test function with known coefficients
3. `detect_1D`: find the active frequencies in one dimension
4. `increment`: glue two dimensions together
5. The full algorithm
6. A function that is not a polynomial

Runs in a few seconds.

## 0. Setup

Install the dependencies first:

```
pip install -r requirements.txt
```

On import, KeOps prints a few startup warnings about CUDA and OpenMP. They are expected on a
machine without a GPU, and nothing in this notebook calls KeOps, so they can be ignored.

The one filter below is unrelated: `mr1lfft/construct_mr1l.py` contains LaTeX like `\in`
inside ordinary strings, which Python 3.12+ reports as a `SyntaxWarning`. It is harmless, but
Jupyter renders warnings on a red background where it reads as an error.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np

REPO = Path.cwd() if (Path.cwd() / "NABOPB.py").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO))

with warnings.catch_warnings():
    warnings.simplefilter("ignore", SyntaxWarning)
    import NABOPB as nb

print(f"numpy {np.__version__}   repo {REPO.name}/   ready")
print("using:", ", ".join(f"nb.{n}" for n in ["detect_1D", "increment", "NABOPB"]))

[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.
[KeOps] Warning : OpenMP library not found, it must be downloaded through Homebrew for apple Silicon chips
[KeOps] Warning : OpenMP support is not available. Disabling OpenMP.


numpy 2.2.6   repo nabopb/   ready
using: nb.detect_1D, nb.increment, nb.NABOPB


## 1. Why not just compute every coefficient?

We want to write $f:[0,1]^d \to \mathbb{C}$ as a short sum

$$f(\mathbf{x}) \;\approx\; \sum_{\mathbf{k}\in I} \hat{f}_{\mathbf{k}}\; e^{2\pi i \langle \mathbf{k},\mathbf{x}\rangle}$$

but we don't know which indices belong in $I$. The obvious plan, computing every
coefficient on a grid $\{-N,\dots,N\}^d$ and keeping the largest, scales like this:

In [2]:
N = 16
print(f"size of the full grid (2N+1)^d, for N = {N}\n")
for d in [1, 2, 4, 8, 14]:
    print(f"   d = {d:>2}   {(2*N+1)**d:>28,d}")
print("\ndemo.py's first example is d = 14, about 1.8e21 coefficients.")

size of the full grid (2N+1)^d, for N = 16

   d =  1                             33
   d =  2                          1,089
   d =  4                      1,185,921
   d =  8              1,406,408,618,241
   d = 14   1,816,331,681,783,800,622,529

demo.py's first example is d = 14, about 1.8e21 coefficients.


So we never build it. Instead:

1. find the active frequencies in **dimension 1 alone** → a small set $I_1$
2. same for **dimension 2 alone** → $I_2$
3. search only the **product** $I_1 \times I_2$, keep the largest $s$
4. glue on dimension 3, prune again, and so on to dimension $d$

Every step stays small. Steps 1 and 2 are [`detect_1D`](NABOPB.py), steps 3 and 4 are
[`increment`](NABOPB.py). Those two functions are the whole algorithm.

## 2. A function whose answer we know

To watch the algorithm work we need ground truth, so we build a trigonometric polynomial
in $d = 6$ with exactly **4** nonzero coefficients. Dimensions 4, 5 and 6 are deliberately
left inert: every active frequency has $k_4 = k_5 = k_6 = 0$.

In [3]:
d = 6
gamma = {"type": "full", "N": 8, "sgn": "default"}

#                k1  k2  k3  k4 k5 k6
freq = np.array([[ 0,  0,  0,  0, 0, 0],
                 [ 3, -2,  1,  0, 0, 0],
                 [-5,  0,  4,  0, 0, 0],
                 [ 1,  7, -3,  0, 0, 0]])
coef = np.array([2.0, 1.0, 0.5, 0.25])

def f(x):
    # x has shape (num_nodes, d) -> one value per node
    return np.transpose(coef @ np.exp(2 * np.pi * 1j * freq @ x.T))

print("ground truth")
for k, c in zip(freq, coef):
    print(f"   k = {str(k):<22}  c = {c}")

print("\nactive frequencies per dimension, i.e. what detect_1D should find:")
for j in range(d):
    print(f"   dim {j+1}: {np.unique(freq[:, j])}")

ground truth
   k = [0 0 0 0 0 0]           c = 2.0
   k = [ 3 -2  1  0  0  0]     c = 1.0
   k = [-5  0  4  0  0  0]     c = 0.5
   k = [ 1  7 -3  0  0  0]     c = 0.25

active frequencies per dimension, i.e. what detect_1D should find:
   dim 1: [-5  0  1  3]
   dim 2: [-2  0  7]
   dim 3: [-3  0  1  4]
   dim 4: [0]
   dim 5: [0]
   dim 6: [0]


Note the only thing NABOPB asks of you:

> `fct_handle(x)` takes an array of shape `(num_nodes, d)` and returns `num_nodes` values.

Everything else (where to sample, which transform to use, how to bookkeep indices) is
the algorithm's business. In `pde_applications/` this same handle is a PDE solver.

## 3. `detect_1D`: one dimension at a time

To study dimension $j$ on its own we need a one-dimensional function. We get one by
freezing every other coordinate at a random **anchor** $\tilde{\mathbf{x}}$:

$$g(t) \;=\; f(\tilde x_1,\dots,\tilde x_{j-1},\; t,\; \tilde x_{j+1},\dots,\tilde x_d)$$

Sample $g$ on an equispaced grid, take a 1-D FFT, keep the largest coefficients. Repeat
with `niter_r` different anchors and union the results.

In [4]:
print(f"search space per dimension: {2*gamma['N']+1} candidates\n")
I_per_dim = {}
for j in range(1, d + 1):
    I, n_samples, n_cands, _ = nb.detect_1D(
        np.array([j]), f, d, gamma,
        basis_flag="Fourier_r1l", sparsity_s_local=20, delta=1e-12, niter_r=5)
    I_per_dim[j] = I
    truth = np.unique(freq[:, j - 1])
    ok = "exact" if np.array_equal(np.sort(I.ravel()), np.sort(truth)) else "MISMATCH"
    print(f"   dim {j}:  found {str(I.ravel()):<16} truth {str(truth):<16} "
          f"{ok:<9} ({n_samples} samples)")

search space per dimension: 17 candidates

dim [1] done, card(I)=4
   dim 1:  found [-5  0  1  3]    truth [-5  0  1  3]    exact     (85 samples)
dim [2] done, card(I)=3
   dim 2:  found [-2  0  7]       truth [-2  0  7]       exact     (85 samples)
dim [3] done, card(I)=4
   dim 3:  found [-3  0  1  4]    truth [-3  0  1  4]    exact     (85 samples)
dim [4] done, card(I)=1
   dim 4:  found [0]              truth [0]              exact     (85 samples)
dim [5] done, card(I)=1
   dim 5:  found [0]              truth [0]              exact     (85 samples)
dim [6] done, card(I)=1
   dim 6:  found [0]              truth [0]              exact     (85 samples)


Two things to notice.

**Dimensions 4, 5 and 6 come back as `[0]`.** They contribute nothing to `f`, so the only
surviving frequency is $k=0$. The algorithm discovered the function's real dimensionality
without being told. In the next step, merging against a set of size 1 is nearly free.

**The anchor is random, and `niter_r` repeats it.** That matters more than it looks: the
coefficient recovered at frequency $k$ is a *sum* over every true frequency whose $j$-th
component equals $k$, each weighted by a phase depending on the anchor. Those terms can
cancel, hiding a genuine index. Repeating with independent anchors makes that
vanishingly unlikely. (Worth a section of its own later.)

## 4. `increment`: gluing two dimensions together

Given detected index sets for two disjoint groups of dimensions, `increment`:

1. builds the candidate set $K = I_a \times I_b$ (the full Cartesian product)
2. samples on $K$, again with random anchors for the dimensions not yet included
3. keeps the largest `sparsity_s_local` coefficients above `delta`

Its input is a dict with four keys, holding two (index set, dimension list) pairs:

| key | meaning |
|---|---|
| `J[(1,1)]` | index set A |
| `J[(1,2)]` | which dimensions A refers to |
| `J[(2,1)]` | index set B |
| `J[(2,2)]` | which dimensions B refers to |

In [5]:
J = {(1, 1): I_per_dim[1], (1, 2): np.array([1]),
     (2, 1): I_per_dim[2], (2, 2): np.array([2])}

I_12, dims_12, val, n_samp, n_cand, _ = nb.increment(
    J, f, d, gamma, "Fourier_r1l",
    sparsity_s=10, sparsity_s_local=10, delta=1e-12, niter_r=5)

print(f"|I_1| = {I_per_dim[1].shape[0]}, |I_2| = {I_per_dim[2].shape[0]}"
      f"  ->  candidate set |K| = {n_cand}")
print(f"survivors: {I_12.shape[0]}, over dims {dims_12}\n")
print("detected (k1, k2) pairs      true (k1, k2) pairs")
for a, b in zip(I_12, np.unique(freq[:, :2], axis=0)):
    print(f"   {str(a):<26} {b}")
print(f"\nval = {val}")

dim-inc of [[1]] and [[2]] done, card(I)=4
|I_1| = 4, |I_2| = 3  ->  candidate set |K| = 12
survivors: 4, over dims [1 2]

detected (k1, k2) pairs      true (k1, k2) pairs
   [-5  0]                    [-5  0]
   [0 0]                      [0 0]
   [1 7]                      [1 7]
   [ 3 -2]                    [ 3 -2]

val = 0


The product shrank from $4 \times 3 = 12$ candidates to 4 survivors, and that ratio
compounds. The next step multiplies *those* 4 against dimension 3, not two raw search
spaces.

`val` is `0`, which is not a bug. While `len(dims) < d` there are still frozen anchor
dimensions contaminating the coefficients, so `increment` returns indices only. Only the
final call, when every dimension is included, has no anchor left and returns real
coefficients. **Coefficients are a by-product of the last step.**

## 5. The whole algorithm

`NABOPB()` is the bookkeeping around what we just did by hand:

```
I  <- detect_1D(dim 1)
for t in 2..d:
    I_t <- detect_1D(dim t)
    I   <- increment(I, I_t)      # merge, prune to s_local
return I, coefficients            # from the final increment
```

In [6]:
result = nb.NABOPB(f, d, "Fourier_r1l", gamma, {"type": "default"}, sparsity_s=10)

print("\n" + "=" * 60)
print(f"{'detected k':<26}{'coefficient':>14}{'true':>8}{'abs err':>11}")
print("-" * 60)
for i in np.argsort(-np.abs(result["val"])):
    k, v = result["index"][i], result["val"][i]
    hit = np.where((freq == k).all(axis=1))[0]
    truth = coef[hit[0]] if len(hit) else 0.0
    print(f"{str(k):<26}{v.real:>14.9f}{truth:>8}{abs(v - truth):>11.1e}")

Algorithm Start: d = 6, basis and method = Fourier_r1l, diminc. approach = default
Algorithm Start: Gamma type = full
Algorithm Start: sparsity s = 10, sparsity s_local = 10, delta = 1.000e-12, det. iterations r = 5
Step 1:
dim [1] done, card(I)=4
Step 2:
dim [2] done, card(I)=3
dim-inc of [[1]] and [[2]] done, card(I)=4
dim [3] done, card(I)=4
dim-inc of [[1 2]] and [[3]] done, card(I)=4
dim [4] done, card(I)=1
dim-inc of [[1 2 3]] and [[4]] done, card(I)=4
dim [5] done, card(I)=1
dim-inc of [[1 2 3 4]] and [[5]] done, card(I)=4
dim [6] done, card(I)=1
Algorithm finished with 4 detected indices.



detected k                   coefficient    true    abs err
------------------------------------------------------------
[0 0 0 0 0 0]                2.000000000     2.0    1.8e-16
[ 3 -2  1  0  0  0]          1.000000000     1.0    2.9e-16
[-5  0  4  0  0  0]          0.500000000     0.5    2.8e-16
[ 1  7 -3  0  0  0]          0.250000000    0.25    8.5e-17


In [7]:
full_grid = (2 * gamma["N"] + 1) ** d
examined = int(np.sum(result["cand_sizes"]))
print(f"samples taken       : {int(np.sum(result['sample_sizes'])):,}")
print(f"candidates examined : {examined:,}")
print(f"full grid would be  : {full_grid:,}")
print(f"                      {full_grid // examined:,}x more")

samples taken       : 955
candidates examined : 142
full grid would be  : 24,137,569
                      169,982x more


## 6. A function that is not a polynomial

Everything so far was a finite trigonometric polynomial, so exact recovery was possible and
the error came out at 1e-16. Real functions have infinitely many nonzero coefficients that
decay, and then sparsity becomes a genuine choice rather than a formality.

`bspline_test_10d` ships with the repo and comes with its exact Fourier coefficients, so we
can still mark our own work. It is a sum of three tensor-product B-splines on **disjoint
groups of variables**:

$$f(\mathbf{x}) = B_2(x_1,x_3,x_8) + B_4(x_2,x_5,x_6,x_{10}) + B_6(x_4,x_7,x_9)$$

Nothing tells NABOPB that this structure exists. Watch it find it anyway.

In [8]:
from bspline_test_10d import bspline_test_10d, bspline_test_10d_fouriercoeff

GROUPS = {"o2": [0, 2, 7], "o4": [1, 4, 5, 9], "o6": [3, 6, 8]}

_, norm2 = bspline_test_10d_fouriercoeff(np.zeros((1, 10), dtype=int))
NORM2 = float(np.ravel(norm2)[0])
print(f"true squared L2 norm of f: {NORM2:.6f}")

print("\ncoefficient decay along one variable of each group:")
print(" " * 13 + "".join([f"k={k}".rjust(11) for k in [1, 3, 5, 7, 9]]))
for name, dim in [("order 2", 0), ("order 4", 1), ("order 6", 3)]:
    row = ""
    for k in [1, 3, 5, 7, 9]:
        fr = np.zeros((1, 10), dtype=int)
        fr[0, dim] = k
        row += f"{abs(bspline_test_10d_fouriercoeff(fr)[0][0]):>11.2e}"
    print(f"   {name:<10}{row}")

true squared L2 norm of f: 3.860521

coefficient decay along one variable of each group:
                     k=1        k=3        k=5        k=7        k=9
   order 2      2.63e-01   2.92e-02   1.05e-02   5.37e-03   3.25e-03
   order 4      1.79e-01   2.21e-03   2.86e-04   7.44e-05   2.72e-05
   order 6      2.09e-01   1.83e-02   1.34e-05   1.77e-06   2.51e-05


Higher spline order means a smoother function means faster decay. The order 6 term is already
down at 1e-5 by $k=5$, while the order 2 term is still at 1e-2. Smoothness buys sparsity,
which is the entire premise of the method.

(We sample odd $k$ because these coefficients vanish exactly at multiples of the spline
order. That is a property of B-splines, not of the decay.)

The order 4 term depends on four variables at once, so no index needs more than four nonzero
entries. That is the **superposition dimension**, and it is why `demo.py` passes
`"superpos": 4`. It removes every candidate with more than four nonzero entries before
sampling. In the run below no candidate has that many, so the key changes nothing there.

In [9]:
import time

gamma_b = {"type": "full", "N": 8, "sgn": "default", "superpos": 4}

np.random.seed(0)
t0 = time.time()
res_b = nb.NABOPB(bspline_test_10d, 10, "Fourier_r1l", gamma_b, {"type": "default"}, 25)

print(f"\n{time.time() - t0:.1f}s, {res_b['index'].shape[0]} indices, "
      f"{int(np.sum(res_b['sample_sizes'])):,} samples")

Algorithm Start: d = 10, basis and method = Fourier_r1l, diminc. approach = default
Algorithm Start: Gamma type = full
Algorithm Start: sparsity s = 25, sparsity s_local = 25, delta = 1.000e-12, det. iterations r = 5
Step 1:
dim [1] done, card(I)=17
Step 2:
dim [2] done, card(I)=17
dim-inc of [[1]] and [[2]] done, card(I)=35
dim [3] done, card(I)=17
dim-inc of [[1 2]] and [[3]] done, card(I)=29
dim [4] done, card(I)=17
dim-inc of [[1 2 3]] and [[4]] done, card(I)=35
dim [5] done, card(I)=17
dim-inc of [[1 2 3 4]] and [[5]] done, card(I)=47
dim [6] done, card(I)=17


dim-inc of [[1 2 3 4 5]] and [[6]] done, card(I)=51
dim [7] done, card(I)=17
dim-inc of [[1 2 3 4 5 6]] and [[7]] done, card(I)=63
dim [8] done, card(I)=17


dim-inc of [[1 2 3 4 5 6 7]] and [[8]] done, card(I)=61
dim [9] done, card(I)=17
dim-inc of [[1 2 3 4 5 6 7 8]] and [[9]] done, card(I)=43
dim [10] done, card(I)=17


Algorithm finished with 25 detected indices.



0.9s, 25 indices, 168,461 samples


In [10]:
from collections import Counter

def which_group(k):
    nz = set(np.flatnonzero(k))
    if not nz:
        return "const"
    for name, g in GROUPS.items():
        if nz <= set(g):
            return name
    return "MIXED"

comp = Counter(which_group(k) for k in res_b["index"])
print("composition of the detected index set:")
for g in ["const", "o2", "o4", "o6", "MIXED"]:
    print(f"   {g:<7} {comp.get(g, 0)}")
print(f"\nindices mixing two different B-splines: {comp.get('MIXED', 0)}")

true_b, _ = bspline_test_10d_fouriercoeff(res_b["index"])
got_b = np.real(res_b["val"])
captured = float(np.sum(np.abs(res_b["val"]) ** 2))
print(f"\ncaptured variance: {captured:.4f} / {NORM2:.4f} = {100 * captured / NORM2:.1f}%")
print(f"max coefficient error at detected indices: {np.max(np.abs(got_b - true_b)):.2e}")

composition of the detected index set:
   const   1
   o2      6
   o4      8
   o6      10
   MIXED   0

indices mixing two different B-splines: 0

captured variance: 2.5030 / 3.8605 = 64.8%
max coefficient error at detected indices: 4.32e-02
